# Naive Bayes

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')


## Load Data

In [ ]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
valid_df = pd.read_csv(os.path.join(data_dir, 'valid_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]


### Danh sách lưu kết quả để xuất CSV

In [ ]:
results_list = []

def evaluate_model(model, name, X_split, y_split, split_name):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)."""
    y_pred = model.predict(X_split)
    y_prob = model.predict_proba(X_split)[:, 1]  # Xác suất class 1 để tính AUC

    acc = accuracy_score(y_split, y_pred)
    prec = precision_score(y_split, y_pred)
    rec = recall_score(y_split, y_pred)
    f1 = f1_score(y_split, y_pred)
    auc = roc_auc_score(y_split, y_prob)

    res = {
        'Algorithm': name,
        'Split': split_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC': round(auc, 4)
    }

    results_list.append(res)
    print(f"[{name} | {split_name}] Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    return res


In [ ]:
# K-FOLD CONFIG
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_kfold(model, name, X_data, y_data, cv_split):
    """Đánh giá K-Fold và lưu kết quả trung bình vào results_list."""
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'auc': 'roc_auc'
    }

    cv_results = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv_split,
        scoring=scoring,
        n_jobs=-1
    )

    res = {
        'Algorithm': name,
        'Split': 'KFold',
        'Accuracy': round(cv_results['test_accuracy'].mean(), 4),
        'Precision': round(cv_results['test_precision'].mean(), 4),
        'Recall': round(cv_results['test_recall'].mean(), 4),
        'F1-Score': round(cv_results['test_f1'].mean(), 4),
        'AUC': round(cv_results['test_auc'].mean(), 4)
    }

    results_list.append(res)
    print(
        f"[{name} | KFold] Acc: {res['Accuracy']:.4f} | Prec: {res['Precision']:.4f} | "
        f"Rec: {res['Recall']:.4f} | F1: {res['F1-Score']:.4f} | AUC: {res['AUC']:.4f}"
    )
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [ ]:
print("--- V1: Baseline Naive Bayes ---")
NaiveBayes_baseline_model = GaussianNB()

# Đánh giá K-Fold trước
evaluate_kfold(NaiveBayes_baseline_model, "V1: Naive Bayes Baseline", X_train, y_train, kfold)

# Train cố định trên train set rồi đánh giá validation/test
NaiveBayes_baseline_model.fit(X_train, y_train)
evaluate_model(NaiveBayes_baseline_model, "V1: Naive Bayes Baseline", X_valid, y_valid, "Validation")
evaluate_model(NaiveBayes_baseline_model, "V1: Naive Bayes Baseline", X_test, y_test, "Test")


### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [ ]:
print("--- V2: GridSearchCV Tuning ---")
param_grid = {'var_smoothing': np.logspace(-12, -7, 6)}

grid_search = GridSearchCV(
    GaussianNB(),
    param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
NaiveBayes_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")

# Đánh giá K-Fold cho mô hình tốt nhất
evaluate_kfold(NaiveBayes_tuned_model, "V2: Naive Bayes Tuned (GridSearch)", X_train, y_train, kfold)

evaluate_model(NaiveBayes_tuned_model, "V2: Naive Bayes Tuned (GridSearch)", X_valid, y_valid, "Validation")
evaluate_model(NaiveBayes_tuned_model, "V2: Naive Bayes Tuned (GridSearch)", X_test, y_test, "Test")


In [ ]:
print("--- V3: Ensemble Learning (Stacking) ---")
# Kết hợp mô hình chính và KNN làm Base Models
base_estimators = [('nb', GaussianNB()), ('knn', KNeighborsClassifier(n_neighbors=5))]

NaiveBayes_stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=GaussianNB(),
    cv=kfold
)

# Đánh giá K-Fold cho stacking
evaluate_kfold(NaiveBayes_stacking_model, "V3: Naive Bayes Stacking Ensemble", X_train, y_train, kfold)

NaiveBayes_stacking_model.fit(X_train, y_train)
evaluate_model(NaiveBayes_stacking_model, "V3: Naive Bayes Stacking Ensemble", X_valid, y_valid, "Validation")
evaluate_model(NaiveBayes_stacking_model, "V3: Naive Bayes Stacking Ensemble", X_test, y_test, "Test")


### (5) Lưu kết quả

In [ ]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/Naive_Bayes/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU 1 FILE CSV DUY NHẤT
df_results = pd.DataFrame(results_list)

csv_output = os.path.join(save_path, 'Naive_Bayes_full_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'Naive_Bayes_baseline.pkl'), 'wb') as f:
    pickle.dump(NaiveBayes_baseline_model, f)

with open(os.path.join(save_path, 'Naive_Bayes_tuned.pkl'), 'wb') as f:
    pickle.dump(NaiveBayes_tuned_model, f)

with open(os.path.join(save_path, 'Naive_Bayes_stacking.pkl'), 'wb') as f:
    pickle.dump(NaiveBayes_stacking_model, f)

print("-" * 40)
print(f"✅ Đã lưu tất cả kết quả vào: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)
